# 리포트 8 — 마이크로도플러 — 도는 로터가 남기는 무늬

> 호버링하는 드론은 제자리에 있지만 **프로펠러는 돈다**. 그 회전이 남기는 시간-주파수 무늬가 이 연구의 분류 축이다.

이 권은 아래 절로 이루어진다. 각 절은 **한 일 · 결과 · 방법 · 재현**을 자기 앞에 달고 있어, 필요한 절만 따로 읽어도 된다.

| 절 | 무엇을 말하나 |
|---|---|
| 1 | 스톡 Paths.doppler 로는 블레이드 변조가 안 나온다 — SceneObject.velocity 가 객체당 강체 1벡터다 |
| 2 | 시간표본마다 자세를 새로 놓고 다시 쏘아 슬로타임 복소열을 만든다 |
| 3 | 두 엔진이 날개끝 주파수 아래에서 겹치고 그 위에서 갈린다 |
| 4 | 네 로터가 같은 회전수로 돌면 무늬는 시간에 못 변한다 |
| 5 | 동체가 날개를 가리면 변조 깊이와 레벨이 함께 바뀐다 |
| 6 | 블레이드 신호는 약하지 않다 — 동체 정적 반사가 덮고 있을 뿐이다 |

⭐ 숫자는 전부 계산 결과 JSON(원장)에서 주입된다 — 각주가 그 출처다. 원장이 다시 계산되면 빌더를 돌리는 것만으로 본문 숫자가 따라 바뀐다.


---

## 절 1. 스톡 Paths.doppler 로는 블레이드 변조가 안 나온다 — SceneObject.velocity 가 객체당 강체 1벡터다



> ### 한 일
> **설치본 Sionna 의 장면 객체에 속도를 직접 넣고 경로 도플러를 읽어 자유도를 셌다.**

### 결과
1. 장면 객체 하나가 갖는 속도 자유도는 3 개 [^1] 다 — 평행이동 세 성분이고, 회전 자유도는 그 밖이다.
2. 정지 장면에서 도플러가 0 이 아닌 경로는 0 개 [^2] (전체 98 개 [^3]) — 배선 자체는 정상이다.
3. 프롭 그룹에 강체속도를 주면 프롭 경유 경로 296 개 [^4] 가 모두 같은 부호로 몰린다 — 크기는 강체 투영에 따라 0 ~ 187.5 Hz [^5] 로 퍼지고, 그 최대가 강체 예측 181.3 Hz [^6] 와 같은 자리에 선다.
4. 전진날과 후퇴날이 갈리려면 그 둘이 반대 부호를 받아야 하는데, 자유도 3 [^1] 개짜리 벡터 하나가 그것을 표현한다.
5. 그래서 남는 길은 하나다 — 시간표본마다 자세를 새로 놓고 다시 쏘는 것이고, 그 절차가 [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) 이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 자유도 세기 | 설치본 `sionna.rt.SceneObject.velocity` 의 성분 수를 부품 객체마다 직접 읽었다 |
| 강체 주입 시험 | 프롭 그룹에만 속도 (0, 0, 30) m/s 를 주고 `Paths.doppler` 의 고유값을 셌다 |
| 대조 | 같은 장면을 정지 상태로 한 번 더 추적해 도플러가 0 인지 확인했다 |
| 무엇을 안 물었나 | Sionna 가 «회전하는 기하의 왕복 위상» 을 따라가는지는 이 시험의 밖이다 — [편 36 «두 엔진»](36_md-two-engines.ipynb) 이 그것을 잰다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_probe.py
```

| | |
|---|---|
| 출력 | `outputs/report15_probe.json` |
| 소요 | 약 7 분 (GPU 1장) |

---


## 무엇을 물었나

Sionna 는 장면 객체에 속도를 주면 경로마다 도플러를 계산해 준다. 그 기능만으로 **블레이드 마이크로도플러**(도는 날개가 만드는 변조)가 나오는지가 첫 갈래다.

나온다면 로터를 돌리는 비싼 길을 건너뛴다. 안 나온다면 시간표본마다 자세를 다시 놓고 광선을 다시 쏘는 길로 간다. 두 길의 비용 차이가 커서 먼저 확인했다.


## 자유도를 세면 답이 정해진다

| 무엇을 | 값 |
|---|---|
| 객체당 속도 자유도 | 3 개 [^1] |
| 정지 장면의 0 아닌 도플러 | 0 개 [^2] |
| 강체속도 주입 시 프롭 경로 | 296 개 [^4] |
| 그 경로의 도플러 최대 | 187.5 Hz [^5] |
| 강체 운동학이 예측하는 값 | 181.3 Hz [^6] |

프롭 전체에 강체속도 하나를 주면 프롭 경유 경로가 전부 같은 부호로 몰리고, 크기는 각 경로의 강체 투영에 따라 0 부터 표의 최대값까지 퍼진다 — 그 최대값이 강체 예측과 같은 자리에 선다. 블레이드 전진/후퇴가 갈라지려면 부호가 반대인 두 값이 나와야 하는데, 이 표의 도플러는 한 부호로 몰린다.


## 이것은 구현 문제가 아니라 자료구조 문제다

속도가 객체당 벡터 하나라는 것은 «그 객체가 강체로 평행이동한다» 는 뜻이다. 회전하는 프로펠러는 같은 객체 안에서 점마다 속도가 다르고, 그 차이를 담을 자리가 벡터 하나 밖에 있다.

부품을 날개 하나하나로 쪼개도 마찬가지다 — 날개 하나 안에서도 뿌리와 끝의 속도가 다르고, 그 차이가 바로 날개끝 확산을 만드는 양이다.

판정 문장은 산출물에 그대로 있다 — `outputs/report15_verdict.json : branch1_paths_doppler.answer`.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 시간표본마다 자세를 새로 놓고 다시 쏜다 | 블레이드 변조를 만드는 유일한 길이 실제로 작동하는지가 갈린다 | [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) |
| Sionna 가 회전 기하의 왕복 위상을 따라가는지 우리 커널과 맞댄다 | 위상을 광선 엔진에 맡길 수 있는지가 정해진다 | [편 36 «두 엔진»](36_md-two-engines.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 6개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.max_dof` | 3 |
| [^2] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.doppler_nonzero_paths_static_scene` | 0 |
| [^3] | `outputs/report15_verdict.json` | `branch1_paths_doppler.evidence.n_paths` | 98 |
| [^4] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.n_target_paths` | 296 |
| [^5] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.doppler_max_hz` | 187.5 |
| [^6] | `outputs/report15_probe.json` | `airframes.matrice4e.A_doppler.rigid_prop_velocity.predicted_rigid_hz` | 181.3 |


---

## 절 2. 시간표본마다 자세를 새로 놓고 다시 쏘아 슬로타임 복소열을 만든다



> ### 한 일
> **로터 위상을 시간표본마다 다시 놓고 광선을 다시 쏘아 되돌아오는 복소 신호의 느린 시간축을 만들었다.**

### 결과
1. 헤드라인 칸은 DJI Matrice 4E [^7] 를 배 쪽에서 본 것이다 — 방위 0 도 [^8] · 앙각 -15 도 [^9].
2. 호버 3800 rpm [^10] 에서 운동학이 예측하는 날개끝 주파수는 1230 Hz [^11], 블레이드 통과율은 126.7 Hz [^12] 다.
3. 표본율 5000 Hz [^13] 로 2526 개 [^14] 를 이어 붙여 창 길이 0.505 s [^15] 를 얻었다.
4. 그 창이 주는 도플러 분해능은 1.98 Hz [^16] 이고, 날개끝까지 621 칸 [^17] 이 든다.
5. 조립을 싸게 만든 것은 `src/articulated_fast.py` 다 — 드론을 한 번 짓고 위상마다 행렬곱만 한다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 슬로타임 복소열 | 시간표본마다 로터 위상을 다시 놓고 광선을 다시 쏜다 — 위상 하나짜리 표를 쓰지 않으므로 로터마다 회전수를 다르게 줄 수 있다 |
| 무엇이 그것을 가능하게 했나 | `src/articulated_fast.py` — 드론을 한 번 짓고 위상마다 행렬곱만 한다. 정점 배열이 옛 함수와 비트 단위로 같다 |
| 도플러 분해능 | 창에 든 블레이드 주기 수가 정한다. 표본 수를 늘려도 안 좋아진다 |
| 헤드라인 기체 선택 | DJI Matrice 4E — 프롭·벨 겹침이 0.01 % 로 정리됐고 1차 실측 표적이다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/build_report15b_figs.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25 분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 34 «스톡 도플러»](34_md-paths-doppler.ipynb) — 왜 이 길로 가나 |

---


## 무엇을 재는가

표적이 제자리에 떠 있어도 날개는 돈다. 날개 표면의 점들이 시간에 따라 자리를 바꾸므로 왕복 위상이 변조되고, 그것이 되돌아오는 신호의 느린 시간축에 실린다. 우리는 그 열을 **시간표본마다 자세를 새로 놓고 광선을 다시 쏘아** 만든다.

왜 그렇게까지 하는가. 로터마다 회전수를 다르게 주려면 드론 전체 자세가 각도 하나의 함수가 아니게 되고, 그러면 «위상 하나짜리 표를 미리 만들어 두고 조회한다» 는 지름길이 막힌다. 조립을 싸게 만들어 그 지름길을 버렸다.


## 헤드라인 칸의 운동학

| 무엇을 | 값 |
|---|---|
| 기체 · 자세 | DJI Matrice 4E [^7] · 배 쪽 |
| 방위 · 앙각 | 0° [^8] · -15° [^9] |
| 호버 회전수 | 3800 rpm [^10] |
| 날개끝 속도 | 54.52 m/s [^18] |
| 날개끝 주파수 | 1230 Hz [^11] |
| 블레이드 통과율 | 126.7 Hz [^12] |


## 창이 분해능을 정한다

| 무엇을 | 값 |
|---|---|
| 표본율 | 5000 Hz [^13] |
| 표본 수 | 2526 개 [^14] |
| 창 길이 | 0.505 s [^15] |
| 도플러 분해능 | 1.98 Hz [^16] |
| 날개끝까지 든 칸 수 | 621 칸 [^17] |

분해능은 «창에 든 블레이드 주기 수» 가 정한다. 같은 창 안에서 표본을 촘촘히 해도 칸이 좁아지지 않으므로, 능선을 가르려면 창을 늘린다.


## 전처리를 어떻게 했는가

마이크로도플러 그림은 전처리가 답을 바꾼다. 그래서 규약을 적어 둔다.

| 단계 | 우리가 한 것 | 왜 |
|---|---|---|
| 채널 | 전체 드론과 프로펠러만을 따로 | 동체가 블레이드를 덮는다 |
| 0 도플러 | **살린다** | 동체 선이 읽기의 기준이다 |
| 조각 길이 | 블레이드 13 주기 | 능선 사이에 13 빈이 들어 빗살이 안 뭉갠다 |
| 창·제로패딩 | Hann · 4배 | 누설을 줄이고 주파수축을 매끈하게 |
| 색역 | 60 dB | 동체 선을 0 dB 로 두고 능선을 그 아래에서 읽는다 |
| 정규화 | 한 그림 안에서 공통 | 두 패널을 나란히 놓고 비교할 수 있게 |


## 검출 축의 전처리는 따로 있다

정적 성분을 지우는 슬로타임 고역통과(MTI)는 `src/microdoppler_proc.py` 에 따로 있다 — **검출 축**에서 쓴다. ⚠ 그 노치는 호버하는 표적의 동체도 함께 지우므로 탐지에서는 대가가 된다.

⚠ 선행 구현의 **처리 파라미터**는 그 시스템의 자원격자에 맞춰진 값이라 그대로 옮기지 않았다. 우리가 가져온 것은 그림을 읽는 순서이고, 차단주파수 같은 것은 우리 물리에서 정했다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 같은 절차를 두 엔진에 태워 무늬를 맞댄다 | 위상을 광선 엔진에 맡기고 세기를 PO 커널에 맡기는 분업이 근거를 얻는다 | [편 36 «두 엔진»](36_md-two-engines.ipynb) |
| 로터마다 회전수를 다르게 준다 | 무늬가 시간에 따라 변하는 데 무엇이 필요한지가 갈린다 | [편 37 «회전수 축»](37_md-rpm.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 12개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^7] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.name` | DJI Matrice 4E |
| [^8] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.az_deg` | 0 |
| [^9] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.el_deg` | -15 |
| [^10] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.rpm` | 3800 |
| [^11] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.f_tip` | 1230 |
| [^12] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.f_flash` | 126.7 |
| [^13] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.prf` | 5000 |
| [^14] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.n_t` | 2526 |
| [^15] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.duration_s` | 0.5053 |
| [^16] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.doppler_resolution_hz` | 1.979 |
| [^17] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.bins_to_ftip` | 621.3 |
| [^18] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.physics.v_tip` | 54.52 |


---

## 절 3. 두 엔진이 날개끝 주파수 아래에서 겹치고 그 위에서 갈린다



> ### 한 일
> **같은 로터 위상 스텝 절차를 스톡 Sionna 엔진과 우리 PO 커널 두 곳에 태워 같은 칸에서 나온 무늬를 맞댔다.**

### 결과
1. 운동학이 예측한 날개끝 주파수 **아래**에서 두 빗살이 겹친다 — 같은 자세·같은 거리에서 변조 깊이가 Sionna 10.91 dB [^19] · PO 커널 17.70 dB [^20] 다.
2. 그 위에서 갈린다 — 기하 절벽 너머 꼬리의 최대값(중앙값)이 Mini 2 에서 Sionna -19.56 dB [^21] 대 PO -36.14 dB [^22] 다.
3. Sionna 의 빗이 **기하에서 온다** — 산란 물리를 전부 빼고 왕복 위상만 더한 기준과 겹치면 근거리에서 Mini 2 9 [^23]/10 [^24] 칸이 ±1 조화 안에서 일치한다.
4. 빗 모양 코사인 중앙값은 Mini 2 0.9639 [^25] · Matrice 4E 0.8714 [^26] 다.
5. ⭐ 아래쪽에서 겹친다는 것이 «위상은 광선 엔진이 맞게 낸다» 의 근거이고, 위쪽에서 갈린다는 것이 «세기는 PO 커널이 맡는다» 의 근거다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 같은 격자 | 로터 위상 스텝·거리·자세·재질·주파수를 전부 같게 두고 엔진 하나만 바꿨다 |
| Sionna 팔 | 스톡 `sionna.rt.PathSolver` — h = Σ a_p·exp(−j2πf_c τ_p). `Paths.cir()` 은 절대위상을 지우므로 쓰지 않는다 |
| PO 팔 | `src/microdoppler_nearfield.py` — 평면파/구면파 PO 표면적분, 가림 없음 |
| 비교 가능한 양만 | 절대 스케일과 기준위상이 다르므로 상수 복소배에 불변인 양(AC 상관 · dB ptp · 하모닉 스펙트럼)만 맞댔다 |
| 기하 기준 | 같은 메쉬로 산란 물리를 빼고 왕복 위상만 더한 빗 — 움직이는 기하가 원리적으로 낼 수 있는 도플러 빗이다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_po_control.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_verdict_geomref.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_verdict.py
```

| | |
|---|---|
| 출력 | `outputs/report15_po_control.json`, `outputs/report15_verdict_geomref.json`, `outputs/report15_verdict.json` |
| 소요 | 약 30 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) — 두 엔진에 태운 절차 |

---


## 같은 로터를 두 엔진이 돌린다

![report07_f2](../outputs/figures/report07_f2.png)

**그림 1.** Sionna 자체 엔진과 우리 PO 커널이 같은 로터에서 같은 무늬를 내는가?
로터 위상을 스텝하고 매번 다시 추적하는 같은 절차를 **서로 다른 두 엔진**에 태웠다 — 하나는 Sionna 의 PathSolver 이고 하나는 우리 PO 커널이다.

운동학이 예측한 날개끝 주파수 **아래**에서 두 빗살이 겹친다.


## 그 위에서 갈린다

| 기체 | Sionna 꼬리 최대(중앙값) | PO 꼬리 | 기하 기준 꼬리 |
|---|---|---|---|
| Mini 2 | -19.56 dB [^21] | -36.14 dB [^22] | -26.52 dB [^27] |
| Matrice 4E | -24.60 dB [^28] | -35.83 dB [^29] | -29.85 dB [^30] |

Sionna 의 꼬리가 PO·기하 기준보다 뚜렷하게 높게 남는다. 그 꼬리는 블레이드가 만든 것이 아니므로, 가장자리를 자동으로 찾는 검출기는 물리적이지 않은 자리를 가장자리라고 보고한다. 그 꼬리의 원인은 [편 42 «광선예산»](42_md-ray-budget.ipynb) 이 잰다.


## 세 엔진을 같은 격자에 태우면

![report07_f5](../outputs/figures/report07_f5.png)

**그림 2.** Sionna·SBR·PO 를 같은 슬로타임 격자에 태우면 같은 무늬가 나오는가?
같은 기체·자세·주파수·PRF·로터별 회전수로 세 엔진을 돌렸다 — 1024 표본 [^31] @ PRF 5000 Hz [^32] = 26 [^33] 블레이드 주기.

| 엔진 | 변조 p-p | 무늬 |
|---|---|---|
| Sionna PathSolver | 9.8 dB [^34] | 능선이 성기고 날개끝 주파수 근처에서 잦아든다 |
| 우리 SBR | 7.5 dB [^35] | 능선이 대역을 채우고 날개끝 주파수 밖에도 남는다 |
| 우리 순수 PO | 2.7 dB [^36] | 능선이 몇 가닥에 그친다 |

⭐ 셋 다 (a) 0 도플러 동체 선, (b) 블레이드 통과율 간격 능선, (c) 날개끝 주파수 근처 감쇠를 낸다 — **구조가 일치한다**. 날개끝 주파수 안에서 스펙트럼 코사인이 0.657 [^37] (Sionna↔SBR) · 0.696 [^38] (Sionna↔PO) · 0.647 [^39] (SBR↔PO) 다.


## 그 코사인은 «닮았다» 까지를 말한다

0.65~0.70 은 구조가 닮았다는 뜻이고, 세 엔진은 능선의 **세기와 밀도**에서 크게 갈린다. 갈리는 방향에 각각 이유가 있다.

| 엔진 | 무엇이 그렇게 만드나 |
|---|---|
| 우리 순수 PO | 가림을 빼고 계산하므로 모든 면이 항상 기여한다 → 변조가 씻긴다 |
| 우리 SBR | ⚠ 날개끝 주파수 밖에도 능선을 낸다 — 운동학이 금지한 자리다 |
| Sionna | 경로가 열 개 남짓이라 표본이 성기다 |

⚠ 남은 의심은 하나다 — **우리 SBR 의 날개끝 밖 능선이 광선 격자의 이산화 산물인가, 물리인가.** [편 42 «광선예산»](42_md-ray-budget.ipynb) 이 그것을 잰다.


## Sionna 의 빗은 기하에서 온다

| 기체 | ±1 조화 안 일치(근거리) | 빗 모양 코사인 중앙값 |
|---|---|---|
| Mini 2 | 9 [^23] / 10 [^24] | 0.9639 [^25] |
| Matrice 4E | 9 [^40] / 10 [^41] | 0.8714 [^26] |

같은 메쉬로 산란 물리를 전부 빼고 왕복 위상만 더한 빗과 Sionna 의 빗을 겹치면 근거리에서 거의 모든 칸이 ±1 조화 안에서 맞는다. 즉 스톡 PathSolver 는 **회전하는 기하의 왕복 위상을 제대로 따라간다** — 그 빗은 몬테카를로 잡음의 산물이 아니다.


## 정반사 경로는 프로펠러에서 사실상 0 이다

자세×로터위상 1,152 칸 [^42] 전수에서 프로펠러에 떨어진 정반사 경로는 0 칸 [^43] 이다.

지금 보이는 무늬는 전부 흩어져 되돌아오는 반사에서 나오고, 그 값은 광선을 다시 쏠 때마다 흔들린다. **위상은 광선 엔진이, 세기는 PO 커널이 맡는** 분업의 근거가 여기 있다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 가장자리 판정을 꼬리 초과분과 함께 다시 낸다 | «가장자리» 가 물리적 절벽인지 꼬리인지가 칸마다 갈린다 | [편 42 «광선예산»](42_md-ray-budget.ipynb) |
| 두 엔진 일치도를 가림 있는 SBR 팔로 한 번 더 잰다 | PO 팔의 «가림 없음» 이 일치도에 넣는 몫이 확정된다 | `src/rcs_sbr.py` 팔을 같은 격자에 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 25개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^19] | `outputs/report00_microdoppler.json` | `rows[1].ptp_sionna_db` | 10.91 |
| [^20] | `outputs/report00_microdoppler.json` | `rows[1].ptp_po_db` | 17.7 |
| [^21] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.sionna_tail_max_db_median` | -19.56 |
| [^22] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.po_tail_max_db_median` | -36.14 |
| [^23] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.mini2.n_near_within_1_bin` | 9 |
| [^24] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.mini2.n_near_cells` | 10 |
| [^25] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.mini2.comb_shape_cosine_median` | 0.9639 |
| [^26] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.matrice4e.comb_shape_cosine_median` | 0.8714 |
| [^27] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.geometry_tail_max_db_median` | -26.52 |
| [^28] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.sionna_tail_max_db_median` | -24.6 |
| [^29] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.po_tail_max_db_median` | -35.83 |
| [^30] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.geometry_tail_max_db_median` | -29.85 |
| [^31] | `outputs/report07_three_engines.json` | `_meta.n` | 1024 |
| [^32] | `outputs/report07_three_engines.json` | `_meta.prf_hz` | 5000 |
| [^33] | `outputs/report07_three_engines.json` | `_meta.blade_periods` | 25.94 |
| [^34] | `outputs/report07_three_engines.json` | `ptp_db.sionna` | 9.782 |
| [^35] | `outputs/report07_three_engines.json` | `ptp_db.sbr` | 7.545 |
| [^36] | `outputs/report07_three_engines.json` | `ptp_db.po` | 2.741 |
| [^37] | `outputs/report07_three_engines.json` | `verdict.cosine_in_ftip.sionna_vs_sbr` | 0.6569 |
| [^38] | `outputs/report07_three_engines.json` | `verdict.cosine_in_ftip.sionna_vs_po` | 0.6962 |
| [^39] | `outputs/report07_three_engines.json` | `verdict.cosine_in_ftip.sbr_vs_po` | 0.6465 |
| [^40] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.matrice4e.n_near_within_1_bin` | 9 |
| [^41] | `outputs/report15_verdict.json` | `geometric_phase_reference.by_airframe.matrice4e.n_near_cells` | 10 |
| [^42] | `outputs/report00_microdoppler.json` | `specular_census.total.n_cells` | 1152 |
| [^43] | `outputs/report00_microdoppler.json` | `specular_census.total.n_with_prop_specular` | 0 |


---

## 절 4. 네 로터가 같은 회전수로 돌면 무늬는 시간에 못 변한다



> ### 한 일
> **네 로터의 회전수를 잠근 팔과 흩뜨린 팔을 같은 격자에서 돌려 창 반쪽 스펙트럼의 상관을 쟀다.**

### 결과
1. 잠근 팔의 반창 스펙트럼 상관은 0.9946 [^44] 다 — 신호가 완전한 주기함수라 스펙트로그램이 창 내내 같은 모습으로 선다.
2. 로터마다 회전수를 0.22% [^45] 흩뜨리면 상관이 0.7713 [^46] 로 내려간다 — 낙차 0.2233 [^47].
3. 그때 변조 깊이도 함께 커진다 — 잠근 팔 4.63 dB [^48] 에서 흩뜨린 팔 8.04 dB [^49] 로.
4. 동체:날개 비도 잠근 팔 2.24 [^50] 에서 흩뜨린 팔 0.20 [^51] 로 내려간다.
5. 흩어짐 폭 ±0.22% [^45] 는 PX4 텔레메트리 실측 산포(모터간 0.07~0.29 %)의 중간값이다 — 우리 표적의 비행 로그가 오면 그 값으로 바꾼다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 단일축 | 잠근 팔과 흩뜨린 팔은 광선 엔진·재질·기하·창 길이가 전부 같고 로터별 회전수만 다르다 |
| 흩뜨림 규약 | 네 로터에 ±0.22% [^45] 를 패턴 [1, −1, −0.55, 0.55] 로 준다 — 무게중심 치우침과 요 토크 균형이 만드는 실제 비대칭을 흉내 낸 것이다 |
| 판정량 | 창을 반으로 갈라 두 스펙트럼의 상관을 잰다. 완전 주기함수면 1 에 붙는다 |
| 흩어짐 폭의 출처 | PX4 텔레메트리 실측(모터간 산포 0.07~0.29 %)의 중간값 ±0.22 % 다 — `_meta.spread_is_declared_ko` 에 출처와 교체 규약을 적었다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/build_report15b_figs.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25 분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) — 이 축이 서는 절차 |

---


## 회전수가 같으면 무늬는 시간에 못 변한다

![report07_f1](../outputs/figures/report07_f1.png)

**그림 1.** 네 로터가 같은 회전수로 돌 때와 흩어질 때, 무늬가 시간에 따라 어떻게 다른가?
왼쪽은 네 로터를 같은 회전수로 돌린 것이다. 줄무늬가 시간축 내내 **같은 자리에 선다**. 창을 반으로 갈라 두 스펙트럼의 상관을 재면 0.9946 [^44] 다.

우연이 아니라 **원리**다 — 네 로터가 같은 속도로 위상까지 맞춰 돌면 신호가 완전한 주기함수가 되고, 주기함수의 스펙트로그램은 창 내내 자기 모습을 지킨다.


## 흩뜨리면 줄이 숨쉰다

| 무엇을 | 잠근 팔 | 흩뜨린 팔 |
|---|---|---|
| 반창 스펙트럼 상관 | 0.9946 [^44] | 0.7713 [^46] |
| 변조 깊이 | 4.63 dB [^48] | 8.04 dB [^49] |
| 동체:날개 비 | 2.24 [^50] | 0.20 [^51] |
| 날개끝 안쪽 에너지 비 | 0.9479 [^52] | 0.9708 [^53] |

오른쪽 열이 «시간에 따라 변한다» 의 정량이다. 상관이 내려간 만큼 줄이 숨쉰다.


## 흩어짐 폭은 실측에서 온다

실제 기체는 무게중심 치우침과 요 토크 균형 때문에 네 모터가 서로 다른 추력을 내고 그만큼 회전수가 갈린다. 그 폭을 PX4 텔레메트리 실측 산포(모터간 0.07~0.29 %)의 중간값 ±0.22% [^45] 로 놓았다 — 출처와 교체 규약은 `outputs/report15b_microdoppler.json : _meta.spread_is_declared_ko` 다.

⭐ 이 편이 말하는 것은 특정 산포값이 아니라 **«무늬가 시간에 변하려면 흩어짐이 필요하다»** 는 구조다 — 실측 크기의 산포에서도 상관이 0.7713 [^46] 로 내려간다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 우리 표적(Matrice 4E)의 비행 로그 모터별 회전수를 넣는다 | PX4 중간값으로 둔 흩어짐 폭이 표적 자신의 측정값으로 바뀐다 | 실측 1차 · Matrice 4E |
| 흩어짐 폭을 사다리로 흔들어 상관 낙차 곡선을 낸다 | «시간 변화» 를 검출기가 쓸 수 있는 최소 흩어짐이 정해진다 | 이 편의 단일축을 격자로 |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 10개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^44] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.locked_half_corr` | 0.9946 |
| [^45] | `outputs/report15b_microdoppler.json` | `_meta.rpm_spread_frac` | 0.0022 |
| [^46] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.spread_half_corr` | 0.7713 |
| [^47] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.findings.rpm_spread_makes_it_time_varying.drop` | 0.2233 |
| [^48] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.modulation_ptp_db` | 4.63 |
| [^49] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.B_sbr_spread.modulation_ptp_db` | 8.038 |
| [^50] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.dc_over_ac` | 2.243 |
| [^51] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.B_sbr_spread.dc_over_ac` | 0.2026 |
| [^52] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.energy_inside_ftip_frac` | 0.9479 |
| [^53] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.B_sbr_spread.energy_inside_ftip_frac` | 0.9708 |


---

## 절 5. 동체가 날개를 가리면 변조 깊이와 레벨이 함께 바뀐다



> ### 한 일
> **광선 엔진·재질·기하·운동학·광선 격자를 전부 같게 두고 «동체가 막느냐» 만 다르게 한 두 팔을 돌렸다.**

### 결과
1. 변조 깊이가 -25.42 dB [^54], 레벨이 +1.04 dB [^55] 바뀐다.
2. 막는 팔의 변조 깊이는 42.47 dB [^56], 안 막는 팔은 67.89 dB [^57] 다.
3. 레벨은 각각 -73.09 dB [^58] 와 -74.12 dB [^59] 다.
4. 두 팔의 광선 격자를 같게 유지했다 — 막는 쪽은 동체를 완전흡수로 두고, 안 막는 쪽은 동체 면만 빼되 정점은 남겼다.
5. ⚠ 부호를 물리로 단정하지 않는다 — 합이 코히런트라 항이 줄어도 레벨이 올라갈 수 있고, 실제로 그런 칸이 있다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 가림 단일축 | 한쪽은 동체를 완전흡수(Γ=0)로 두어 막되 산란은 안 하게 하고, 다른 쪽은 동체 면만 빼되 정점은 남겨 광선 격자를 같게 유지한다 |
| 정점을 남기는 이유 | 경계상자가 같아야 두 팔의 광선 수와 간격이 같아진다. 정점을 빼면 «가림» 과 «표본화» 가 섞인다 |
| 채널 | 블레이드 채널만 본다 — 동체 정적 반사를 빼야 가림의 효과가 드러난다 |
| 부호 규약 | 산출물이 부호 주의를 함께 적는다 — `findings.occlusion_sign_note_ko` |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/build_report15b_figs.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25 분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 35 «슬로타임 복소열»](35_md-slowtime.ipynb) — 이 축이 서는 절차 |

---


## 가림만 남기고 다른 것을 전부 묶었다

![report07_f4](../outputs/figures/report07_f4.png)

**그림 1.** 같은 광선·같은 메쉬·같은 운동에서 동체가 막으면 무엇이 달라지는가?
한쪽은 동체를 완전흡수로 두어 **광선은 막되 산란은 안 하게** 하고, 다른 쪽은 동체 면만 빼되 정점 배열은 남겨 두었다.

정점을 남기면 경계상자가 같아서 두 팔의 광선 수와 간격이 같아진다. 정점을 빼면 «가림» 과 «표본화» 가 섞여 축이 둘이 된다.


## 두 팔의 값

| 무엇을 | 막는 팔 | 안 막는 팔 | 차이 |
|---|---|---|---|
| 변조 깊이 | 42.47 dB [^56] | 67.89 dB [^57] | -25.42 dB [^54] |
| 레벨 | -73.09 dB [^58] | -74.12 dB [^59] | +1.04 dB [^55] |
| 동체:날개 비 | 0.063 [^60] | 0.090 [^61] | — |
| 날개끝 안쪽 에너지 비 | 0.8267 [^62] | 0.8252 [^63] | — |


## 부호를 물리로 단정하지 않는 이유

합이 코히런트라 항이 줄어도 남은 항끼리 상쇄가 덜 되면 레벨이 **올라갈 수 있다**. 실제로 그런 칸이 있다.

근거는 `outputs/report15b_microdoppler.json : cells.*.findings.occlusion_sign_note_ko` 에 있다. 그래서 이 편이 말하는 것은 «가림이 레벨을 내린다» 가 아니라 **«가림이 두 양을 함께 움직인다»** 다.

가림이 무는 자세가 어디인지는 [편 40 «자세 축»](40_md-attitude.ipynb) 이 잰다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 가림을 자세 전면으로 넓힌다 | 어느 자세에서 얼마나 무는지의 지도가 서고, 그 지도가 분류기의 입력이 된다 | 이 편의 단일축을 격자로 |
| 가림 있는 커널을 사다리 채점에 정식 편입한다 | 형상 사다리의 부호가 가림 아래에서 유지되는지가 갈린다 | [편 33 «사다리 판정 조건»](33_ladder-premature.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 10개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^54] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.findings.occlusion_ptp_db` | -25.42 |
| [^55] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.findings.occlusion_level_db` | 1.038 |
| [^56] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.modulation_ptp_db` | 42.47 |
| [^57] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.G_blade_free.modulation_ptp_db` | 67.89 |
| [^58] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.level_db` | -73.09 |
| [^59] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.G_blade_free.level_db` | -74.12 |
| [^60] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.dc_over_ac` | 0.06278 |
| [^61] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.G_blade_free.dc_over_ac` | 0.09026 |
| [^62] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.energy_inside_ftip_frac` | 0.8267 |
| [^63] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.G_blade_free.energy_inside_ftip_frac` | 0.8252 |


---

## 절 6. 블레이드 신호는 약하지 않다 — 동체 정적 반사가 덮고 있을 뿐이다



> ### 한 일
> **같은 자세·같은 주파수에서 전체 드론 채널과 프로펠러 채널을 따로 재어 변조 깊이를 맞댔다.**

### 결과
1. 전체 드론 채널의 변조 깊이는 4.63 dB [^64] 인데, 같은 칸의 프로펠러 채널은 42.47 dB [^65] 다.
2. 두 채널의 레벨은 -50.43 dB [^66] 와 -73.09 dB [^67] 로, 동체가 훨씬 밝다.
3. 동체:날개 비가 전체 채널에서 2.24 [^68], 프로펠러 채널에서 0.063 [^69] 다.
4. ⭐ 블레이드 신호가 약한 것이 아니라 **동체 정적 반사가 덮고 있다** — 그것이 전처리에서 정적 성분을 지우는 이유다.
5. 어려운 것은 «블레이드가 약하다» 가 아니라 **«동체와 블레이드를 가르는 일»** 이다.


### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 채널 분리 | 같은 추적에서 프로펠러 부품을 지난 경로만 따로 모아 프로펠러 채널을 만든다 — 기하·재질·광선 격자는 전체 채널과 같다 |
| 변조 깊이 | 슬로타임 \|h\| 의 최대−최소 [dB]. 동체 정적 반사가 크면 이 값이 눌린다 |
| 동체:날개 비 | DC(정적 성분) 대 AC(변조 성분) 의 비. 이 값이 클수록 동체가 덮는다 |
| 무엇을 안 주장하나 | 절대 레벨은 이 축의 산출이 아니다 — 두 채널의 **비**만 읽는다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_microdoppler_recompute.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15b_stamp_provenance.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/build_report15b_figs.py
```

| | |
|---|---|
| 출력 | `outputs/report15b_microdoppler.json`, `outputs/report15b_series.npz` |
| 소요 | 약 25 분 (GPU 1장 — 광선 추적이 6칸 × 4팔) |
| 비고 | 산출물이 자기가 어떤 메쉬로 계산됐는지 지문을 함께 적는다(`mesh_provenance`) — 계산 도중 메쉬가 바뀌면 스스로 경고한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 38 «가림 축»](38_md-occlusion.ipynb) — 두 팔의 정의 |

---


## 같은 칸을 두 채널로 읽는다

![report07_f3](../outputs/figures/report07_f3.png)

**그림 1.** 블레이드 신호는 약한가, 아니면 동체가 덮고 있는가?
같은 자세·같은 주파수에서 전체 드론과 프로펠러 채널을 따로 쟀다. 두 채널은 같은 추적에서 나오므로 기하·재질·광선 격자가 같다.


## 두 채널의 값

| 무엇을 | 전체 드론 채널 | 프로펠러 채널 |
|---|---|---|
| 변조 깊이 | 4.63 dB [^64] | 42.47 dB [^65] |
| 레벨 | -50.43 dB [^66] | -73.09 dB [^67] |
| 동체:날개 비 | 2.24 [^68] | 0.063 [^69] |
| 반창 스펙트럼 상관 | 0.9946 [^70] | 0.7295 [^71] |

⭐ 프로펠러 채널의 변조는 전체 채널보다 열 배 넘게 깊다. 레벨은 반대로 훨씬 낮다. 두 줄을 같이 읽으면 **동체가 밝아서 변조를 눌렀다** 가 나온다.


## 그래서 무엇이 어려운가

전기적으로 보면 블레이드 폭은 우리 대역에서 파장의 한 자릿수 분율이다. 그런데도 프로펠러 채널의 변조 자체는 42.5 dB [^65] 로 깊다.

어려운 것은 «블레이드가 약하다» 가 아니라 **«동체와 블레이드를 가르는 일»** 이다. 검출 축에서 정적 성분을 지우는 이유가 여기 있고, 그 노치가 호버 표적의 동체까지 지우는 대가도 여기서 나온다.

⚠ PO 커널이 블레이드 폭에서 얼마나 약한지는 [편 33 «사다리 판정 조건»](33_ladder-premature.ipynb) 이 dB 로 적었다.


## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 정적 성분 제거 뒤의 블레이드 채널로 검출을 돌린다 | «동체를 지우는 대가» 와 «블레이드를 얻는 이득» 이 같은 저울에 올라간다 | [편 43 «상시 신호의 상한»](43_md-prf.ipynb) |
| 동체:날개 비를 자세 격자로 넓힌다 | 어느 자세에서 블레이드가 드러나는지의 지도가 선다 | [편 40 «자세 축»](40_md-attitude.ipynb) |


<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 8개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^64] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.modulation_ptp_db` | 4.63 |
| [^65] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.modulation_ptp_db` | 42.47 |
| [^66] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.level_db` | -50.43 |
| [^67] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.level_db` | -73.09 |
| [^68] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.dc_over_ac` | 2.243 |
| [^69] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.dc_over_ac` | 0.06278 |
| [^70] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.A_sbr_locked.half_window_spectrum_corr` | 0.9946 |
| [^71] | `outputs/report15b_microdoppler.json` | `cells.matrice4e/belly.arms.F_blade_occ.half_window_spectrum_corr` | 0.7295 |
